<a href="https://colab.research.google.com/github/eltongaspar/python/blob/Advpl/Aula_12_Pre_processamento_Texto_Aluno_ProEducador.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Aula 12 – Pré-processamento de Texto  
## Deep Learning e Inteligência Artificial Generativa Aplicada

### Versão do Aluno — com campos para preenchimento

**Tema da aula:** pré-processamento textual para projetos de NLP.

Nesta aula, você irá aplicar etapas fundamentais de preparação de textos para projetos de NLP.

### Capacidades trabalhadas
- Desenvolver soluções com NLP.
- Organizar pipeline de pré-processamento textual.
- Demonstrar coerência técnica na limpeza textual.

### Conhecimentos
- Tokenização.
- Stopwords.
- Stemming.
- Lematização.
- Uso de NLTK, SpaCy e Google Colab.

### Evidência de aprendizagem
Notebook de NLP concluído, contendo leitura do dataset, limpeza textual, tokenização, remoção de stopwords, stemming, lematização e comparação entre os resultados.

## 1. Contextualização da aula

Em projetos de **NLP**, os textos precisam ser preparados antes de serem utilizados em modelos de IA.

Nesta aula, você trabalhará com um dataset de registros industriais simulados e deverá construir um pipeline de pré-processamento textual.

In [1]:
# ============================================================
# 2. Instalação e importação das bibliotecas
# ============================================================

# Em ambiente Google Colab, execute as linhas abaixo se necessário:
# !pip install -q nltk spacy
# !python -m spacy download pt_core_news_sm

import pandas as pd
import re
import string
import nltk
import spacy

from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import RSLPStemmer

nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download('stopwords')
nltk.download('rslp')

print("Bibliotecas carregadas com sucesso!")

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.


Bibliotecas carregadas com sucesso!


[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.
[nltk_data] Downloading package rslp to /root/nltk_data...
[nltk_data]   Unzipping stemmers/rslp.zip.


In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## 3. Carregamento do dataset

Faça upload do arquivo:

`aula12_dataset_textos_industriais.csv`

Depois, carregue o arquivo usando Pandas.

In [3]:
# Caso esteja no Colab, faça upload do arquivo CSV antes de executar esta célula.
# from google.colab import files
# files.upload()

# TODO: carregue o dataset com pd.read_csv()
df = pd.read_csv('/content/drive/MyDrive/Classroom/Deep Learning Pró Educador/aula12_dataset_textos_industriais.csv')

# TODO: visualize as primeiras linhas
df.head(10)

,id,texto_original,categoria,sentimento
0,1,Foram identificadas peças com rebarbas e dimen...,qualidade,negativo
1,2,"Após ajuste no sensor de corrente, o sistema v...",manutencao,positivo
2,3,A ANÁLISE DE CONSUMO PERMITIU REDUZIR DESPERDÍ...,energia,positivo
3,4,O ALMOXARIFADO SEPAROU OS MATERIAIS NECESSÁRIO...,logistica,neutro
4,5,A otimização do setup reduziu o tempo de ciclo...,producao,positivo
5,6,A campanha de segurança reduziu ocorrências de...,seguranca,positivo
6,7,Foram identificadas peças com rebarbas e dimen...,qualidade,negativo
7,8,"Após ajuste no sensor de corrente, o sistema v...",manutencao,positivo
8,9,A análise de consumo permitiu reduzir desperdí...,energia,positivo
9,10,"Após ajuste no sensor de corrente, o sistema v...",manutencao,positivo


In [6]:
# TODO: verifique as informações gerais do dataset
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 120 entries, 0 to 119
Data columns (total 4 columns):
 #   Column          Non-Null Count  Dtype 
---  ------          --------------  ----- 
 0   id              120 non-null    int64 
 1   texto_original  120 non-null    object
 2   categoria       120 non-null    object
 3   sentimento      120 non-null    object
dtypes: int64(1), object(3)
memory usage: 3.9+ KB


In [7]:
# TODO: visualize a distribuição da coluna categoria
df['categoria'].value_counts()

,count
categoria,
energia,29
manutencao,21
qualidade,20
producao,20
logistica,16
seguranca,14


In [8]:
# TODO: visualize alguns textos originais
for i, texto in enumerate(df['texto_original'].head(10), start=1):
    print(f"{i}. {texto}")

1. Foram identificadas peças com rebarbas e dimensões fora da tolerância no lote L2026-15....
2. Após ajuste no sensor de corrente, o sistema voltou a operar com estabilidade na linha Robô 2. às 08h
3. A ANÁLISE DE CONSUMO PERMITIU REDUZIR DESPERDÍCIOS DE ENERGIA NO PROCESSO INDUSTRIAL.
4. O ALMOXARIFADO SEPAROU OS MATERIAIS NECESSÁRIOS PARA A ORDEM DE PRODUÇÃO OP-2088. SENSOR_OK
5. A otimização do setup reduziu o tempo de ciclo e aumentou a produtividade da linha A.
6. A campanha de segurança reduziu ocorrências de quase acidente na área de montagem.
7. Foram identificadas peças com rebarbas e dimensões fora da tolerância no lote L2026-30. #urgente
8. Após ajuste no sensor de corrente, o sistema voltou a operar com estabilidade na linha Robô 2. SENSOR_OK
9. A análise de consumo permitiu reduzir desperdícios de energia no processo industrial.
10. Após ajuste no sensor de corrente, o sistema voltou a operar com estabilidade na linha B. operador informou


## 4. Limpeza inicial do texto

Agora, crie uma função para limpar o texto.

A função deverá:

1. converter o texto para letras minúsculas;
2. remover números;
3. remover pontuação;
4. remover espaços extras.

In [9]:
def limpar_texto(texto):
    """
    Função para limpeza inicial do texto.
    Complete os espaços indicados.
    """
    texto = str(texto).lower()
    texto = re.sub(r'\d+', ' ', texto)
    texto = texto.translate(str.maketrans('', '', string.punctuation))
    texto = re.sub(r'\s+', ' ', texto).strip()
    return texto

# TODO: aplique a função na coluna de texto original
df['texto_limpo'] = df['texto_original'].apply(limpar_texto)

# TODO: visualize o texto original e o texto limpo
df[['texto_original', 'texto_limpo']].head(10)

,texto_original,texto_limpo
0,Foram identificadas peças com rebarbas e dimen...,foram identificadas peças com rebarbas e dimen...
1,"Após ajuste no sensor de corrente, o sistema v...",após ajuste no sensor de corrente o sistema vo...
2,A ANÁLISE DE CONSUMO PERMITIU REDUZIR DESPERDÍ...,a análise de consumo permitiu reduzir desperdí...
3,O ALMOXARIFADO SEPAROU OS MATERIAIS NECESSÁRIO...,o almoxarifado separou os materiais necessário...
4,A otimização do setup reduziu o tempo de ciclo...,a otimização do setup reduziu o tempo de ciclo...
5,A campanha de segurança reduziu ocorrências de...,a campanha de segurança reduziu ocorrências de...
6,Foram identificadas peças com rebarbas e dimen...,foram identificadas peças com rebarbas e dimen...
7,"Após ajuste no sensor de corrente, o sistema v...",após ajuste no sensor de corrente o sistema vo...
8,A análise de consumo permitiu reduzir desperdí...,a análise de consumo permitiu reduzir desperdí...
9,"Após ajuste no sensor de corrente, o sistema v...",após ajuste no sensor de corrente o sistema vo...


## 5. Tokenização

Tokenização é o processo de dividir um texto em unidades menores chamadas tokens.

Complete a função abaixo.

In [10]:
def tokenizar_texto(texto):
    return word_tokenize(texto, language='portuguese')

# TODO: aplique a função de tokenização
df['tokens'] = df['texto_limpo'].apply(tokenizar_texto)

df[['texto_limpo', 'tokens']].head(10)

,texto_limpo,tokens
0,foram identificadas peças com rebarbas e dimen...,"[foram, identificadas, peças, com, rebarbas, e..."
1,após ajuste no sensor de corrente o sistema vo...,"[após, ajuste, no, sensor, de, corrente, o, si..."
2,a análise de consumo permitiu reduzir desperdí...,"[a, análise, de, consumo, permitiu, reduzir, d..."
3,o almoxarifado separou os materiais necessário...,"[o, almoxarifado, separou, os, materiais, nece..."
4,a otimização do setup reduziu o tempo de ciclo...,"[a, otimização, do, setup, reduziu, o, tempo, ..."
5,a campanha de segurança reduziu ocorrências de...,"[a, campanha, de, segurança, reduziu, ocorrênc..."
6,foram identificadas peças com rebarbas e dimen...,"[foram, identificadas, peças, com, rebarbas, e..."
7,após ajuste no sensor de corrente o sistema vo...,"[após, ajuste, no, sensor, de, corrente, o, si..."
8,a análise de consumo permitiu reduzir desperdí...,"[a, análise, de, consumo, permitiu, reduzir, d..."
9,após ajuste no sensor de corrente o sistema vo...,"[após, ajuste, no, sensor, de, corrente, o, si..."


In [11]:
# TODO: crie uma coluna com a quantidade de tokens
df['qtd_tokens'] = df['tokens'].apply(len)

df[['texto_limpo', 'qtd_tokens']].head(10)

,texto_limpo,qtd_tokens
0,foram identificadas peças com rebarbas e dimen...,13
1,após ajuste no sensor de corrente o sistema vo...,18
2,a análise de consumo permitiu reduzir desperdí...,12
3,o almoxarifado separou os materiais necessário...,13
4,a otimização do setup reduziu o tempo de ciclo...,16
5,a campanha de segurança reduziu ocorrências de...,13
6,foram identificadas peças com rebarbas e dimen...,14
7,após ajuste no sensor de corrente o sistema vo...,17
8,a análise de consumo permitiu reduzir desperdí...,12
9,após ajuste no sensor de corrente o sistema vo...,18


## 6. Stopwords

Stopwords são palavras muito frequentes que podem carregar pouca informação para algumas tarefas de NLP.

Agora, carregue as stopwords em português e remova essas palavras dos tokens.

In [12]:
# TODO: carregue as stopwords em português
stopwords_pt = set(stopwords.words('portuguese'))

print("Quantidade de stopwords em português:", len(stopwords_pt))
print(list(stopwords_pt)[:30])

Quantidade de stopwords em português: 207
['esta', 'seremos', 'o', 'você', 'tiverem', 'tua', 'estou', 'aquele', 'te', 'teu', 'estavam', 'numa', 'forem', 'no', 'suas', 'com', 'sejam', 'delas', 'tivéssemos', 'num', 'nas', 'deles', 'seríamos', 'tuas', 'a', 'está', 'para', 'vos', 'tivermos', 'teria']


In [13]:
def remover_stopwords(tokens):
    tokens_filtrados = [token for token in tokens if token not in stopwords_pt]
    return tokens_filtrados

# TODO: aplique a remoção de stopwords
df['tokens_sem_stopwords'] = df['tokens'].apply(remover_stopwords)

df[['tokens', 'tokens_sem_stopwords']].head(10)

,tokens,tokens_sem_stopwords
0,"[foram, identificadas, peças, com, rebarbas, e...","[identificadas, peças, rebarbas, dimensões, to..."
1,"[após, ajuste, no, sensor, de, corrente, o, si...","[após, ajuste, sensor, corrente, sistema, volt..."
2,"[a, análise, de, consumo, permitiu, reduzir, d...","[análise, consumo, permitiu, reduzir, desperdí..."
3,"[o, almoxarifado, separou, os, materiais, nece...","[almoxarifado, separou, materiais, necessários..."
4,"[a, otimização, do, setup, reduziu, o, tempo, ...","[otimização, setup, reduziu, tempo, ciclo, aum..."
5,"[a, campanha, de, segurança, reduziu, ocorrênc...","[campanha, segurança, reduziu, ocorrências, qu..."
6,"[foram, identificadas, peças, com, rebarbas, e...","[identificadas, peças, rebarbas, dimensões, to..."
7,"[após, ajuste, no, sensor, de, corrente, o, si...","[após, ajuste, sensor, corrente, sistema, volt..."
8,"[a, análise, de, consumo, permitiu, reduzir, d...","[análise, consumo, permitiu, reduzir, desperdí..."
9,"[após, ajuste, no, sensor, de, corrente, o, si...","[após, ajuste, sensor, corrente, sistema, volt..."


In [19]:
df['qtd_tokens_sem_stopwords'] = df['tokens_sem_stopwords'].apply(len)

reducao_media = (df['qtd_tokens'] - df['qtd_tokens_sem_stopwords']) .mean ()
print (f'Redução média de tokens: {reducao_media :.1f} por registro')

df[['qtd_tokens', 'qtd_tokens_sem_stopwords']].head(10)

Redução média de tokens: 5.0 por registro


,qtd_tokens,qtd_tokens_sem_stopwords
0,13,7
1,18,11
2,12,8
3,13,8
4,16,8
5,13,8
6,14,8
7,17,11
8,12,8
9,18,12


## 7. Stemming

O stemming reduz palavras para um radical aproximado.

Nesta etapa, utilize o `RSLPStemmer` do NLTK.

In [20]:
# TODO: instancie o stemmer
stemmer = RSLPStemmer()

def aplicar_stemming(tokens):
    return [stemmer.stem(token) for token in tokens]

# TODO: aplique stemming nos tokens sem stopwords
df['tokens_stemming'] = df['tokens_sem_stopwords'].apply(aplicar_stemming)

df[['tokens_sem_stopwords', 'tokens_stemming']].head(10)

,tokens_sem_stopwords,tokens_stemming
0,"[identificadas, peças, rebarbas, dimensões, to...","[identific, peç, rebarb, dimens, toler, lot, l]"
1,"[após, ajuste, sensor, corrente, sistema, volt...","[após, ajust, sensor, corr, sistem, volt, oper..."
2,"[análise, consumo, permitiu, reduzir, desperdí...","[anális, consum, permit, reduz, desperdíci, en..."
3,"[almoxarifado, separou, materiais, necessários...","[almoxarif, separ, mater, necess, ord, produç,..."
4,"[otimização, setup, reduziu, tempo, ciclo, aum...","[otimiz, setup, reduz, temp, cicl, aument, pro..."
5,"[campanha, segurança, reduziu, ocorrências, qu...","[campanh, seguranç, reduz, ocorr, quas, acid, ..."
6,"[identificadas, peças, rebarbas, dimensões, to...","[identific, peç, rebarb, dimens, toler, lot, l..."
7,"[após, ajuste, sensor, corrente, sistema, volt...","[após, ajust, sensor, corr, sistem, volt, oper..."
8,"[análise, consumo, permitiu, reduzir, desperdí...","[anális, consum, permit, reduz, desperdíci, en..."
9,"[após, ajuste, sensor, corrente, sistema, volt...","[após, ajust, sensor, corr, sistem, volt, oper..."


In [21]:
# TODO: teste o stemming com palavras de exemplo
palavras_exemplo = ['operação', 'operando', 'operacional', 'produção', 'produtividade', 'manutenção', 'preventiva']

for palavra in palavras_exemplo:
    print(f"{palavra:15} -> {stemmer.stem(palavra)}")

operação        -> oper
operando        -> oper
operacional     -> operac
produção        -> produç
produtividade   -> produt
manutenção      -> manutenç
preventiva      -> preven


## 8. Lematização com SpaCy

A lematização reduz uma palavra à sua forma base, chamada lema.

Agora, carregue o modelo de português do SpaCy e crie uma função de lematização.

In [23]:
!python -m spacy download pt_core_news_sm

try:
    nlp = spacy.load("pt_core_news_sm")
    print("Modelo SpaCy carregado com sucesso!")
except OSError:
    print("Modelo pt_core_news_sm não encontrado.")
    print("No Colab, execute: !python -m spacy download pt_core_news_sm")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.0/13.0 MB 109.5 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('pt_core_news_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.
Modelo SpaCy carregado com sucesso!


In [24]:
def lematizar_texto(texto):
    doc = nlp(texto)
    lemas = [
        token.lemma_
        for token in doc
        if token.text not in stopwords_pt and not token.is_punct and not token.is_space
    ]
    return lemas

# TODO: aplique a função de lematização
df['tokens_lematizados'] = df['texto_limpo'].apply(lematizar_texto)

df[['texto_limpo', 'tokens_lematizados']].head(10)

,texto_limpo,tokens_lematizados
0,foram identificadas peças com rebarbas e dimen...,"[identificar, peça, rebarba, dimensão, tolerân..."
1,após ajuste no sensor de corrente o sistema vo...,"[após, ajustir, sensor, corrente, sistema, vol..."
2,a análise de consumo permitiu reduzir desperdí...,"[análise, consumo, permitir, reduzir, desperdí..."
3,o almoxarifado separou os materiais necessário...,"[almoxarifado, separar, material, necessário, ..."
4,a otimização do setup reduziu o tempo de ciclo...,"[otimização, setup, reduzir, tempo, ciclo, aum..."
5,a campanha de segurança reduziu ocorrências de...,"[campanha, segurança, reduzir, ocorrência, qua..."
6,foram identificadas peças com rebarbas e dimen...,"[identificar, peça, rebarba, dimensão, tolerân..."
7,após ajuste no sensor de corrente o sistema vo...,"[após, ajustir, sensor, corrente, sistema, vol..."
8,a análise de consumo permitiu reduzir desperdí...,"[análise, consumo, permitir, reduzir, desperdí..."
9,após ajuste no sensor de corrente o sistema vo...,"[após, ajustir, sensor, corrente, sistema, vol..."


## 9. Comparação entre texto original, stemming e lematização

Compare os resultados das técnicas aplicadas.

In [25]:
comparacao = df[['texto_original', 'texto_limpo', 'tokens_sem_stopwords', 'tokens_stemming', 'tokens_lematizados']].head(8)

comparacao

,texto_original,texto_limpo,tokens_sem_stopwords,tokens_stemming,tokens_lematizados
0,Foram identificadas peças com rebarbas e dimen...,foram identificadas peças com rebarbas e dimen...,"[identificadas, peças, rebarbas, dimensões, to...","[identific, peç, rebarb, dimens, toler, lot, l]","[identificar, peça, rebarba, dimensão, tolerân..."
1,"Após ajuste no sensor de corrente, o sistema v...",após ajuste no sensor de corrente o sistema vo...,"[após, ajuste, sensor, corrente, sistema, volt...","[após, ajust, sensor, corr, sistem, volt, oper...","[após, ajustir, sensor, corrente, sistema, vol..."
2,A ANÁLISE DE CONSUMO PERMITIU REDUZIR DESPERDÍ...,a análise de consumo permitiu reduzir desperdí...,"[análise, consumo, permitiu, reduzir, desperdí...","[anális, consum, permit, reduz, desperdíci, en...","[análise, consumo, permitir, reduzir, desperdí..."
3,O ALMOXARIFADO SEPAROU OS MATERIAIS NECESSÁRIO...,o almoxarifado separou os materiais necessário...,"[almoxarifado, separou, materiais, necessários...","[almoxarif, separ, mater, necess, ord, produç,...","[almoxarifado, separar, material, necessário, ..."
4,A otimização do setup reduziu o tempo de ciclo...,a otimização do setup reduziu o tempo de ciclo...,"[otimização, setup, reduziu, tempo, ciclo, aum...","[otimiz, setup, reduz, temp, cicl, aument, pro...","[otimização, setup, reduzir, tempo, ciclo, aum..."
5,A campanha de segurança reduziu ocorrências de...,a campanha de segurança reduziu ocorrências de...,"[campanha, segurança, reduziu, ocorrências, qu...","[campanh, seguranç, reduz, ocorr, quas, acid, ...","[campanha, segurança, reduzir, ocorrência, qua..."
6,Foram identificadas peças com rebarbas e dimen...,foram identificadas peças com rebarbas e dimen...,"[identificadas, peças, rebarbas, dimensões, to...","[identific, peç, rebarb, dimens, toler, lot, l...","[identificar, peça, rebarba, dimensão, tolerân..."
7,"Após ajuste no sensor de corrente, o sistema v...",após ajuste no sensor de corrente o sistema vo...,"[após, ajuste, sensor, corrente, sistema, volt...","[após, ajust, sensor, corr, sistem, volt, oper...","[após, ajustir, sensor, corrente, sistema, vol..."


In [26]:
indice = 0

print("Texto original:")
print(df.loc[indice, 'texto_original'])

print("\nTexto limpo:")
print(df.loc[indice, 'texto_limpo'])

print("\nTokens sem stopwords:")
print(df.loc[indice, 'tokens_sem_stopwords'])

print("\nStemming:")
print(df.loc[indice, 'tokens_stemming'])

print("\nLematização:")
print(df.loc[indice, 'tokens_lematizados'])

Texto original:
Foram identificadas peças com rebarbas e dimensões fora da tolerância no lote L2026-15....

Texto limpo:
foram identificadas peças com rebarbas e dimensões fora da tolerância no lote l

Tokens sem stopwords:
['identificadas', 'peças', 'rebarbas', 'dimensões', 'tolerância', 'lote', 'l']

Stemming:
['identific', 'peç', 'rebarb', 'dimens', 'toler', 'lot', 'l']

Lematização:
['identificar', 'peça', 'rebarba', 'dimensão', 'tolerância', 'lote', 'l']


## 10. Construção de um pipeline completo

Agora, organize as etapas em funções de pipeline.

Pipeline com NLTK:

1. limpar texto;
2. tokenizar;
3. remover stopwords;
4. aplicar stemming.

Pipeline com SpaCy:

1. limpar texto;
2. lematizar.

In [28]:
def pipeline_nltk_stemming(texto):
    texto_limpo = limpar_texto(texto)
    tokens = tokenizar_texto(texto_limpo)
    tokens = remover_stopwords(tokens)
    tokens_stem = aplicar_stemming(tokens)
    return tokens_stem

def pipeline_spacy_lematizacao(texto):
    texto_limpo = limpar_texto(texto)
    tokens_lematizados = lematizar_texto(texto_limpo)
    return tokens_lematizados

# TODO: aplique os pipelines
df['pipeline_stemming'] = df['texto_original'].apply(pipeline_nltk_stemming)
df['pipeline_lematizacao'] = df['texto_original'].apply(pipeline_spacy_lematizacao)

df[['texto_original', 'pipeline_stemming', 'pipeline_lematizacao']].head(10)

,texto_original,pipeline_stemming,pipeline_lematizacao
0,Foram identificadas peças com rebarbas e dimen...,"[identific, peç, rebarb, dimens, toler, lot, l]","[identificar, peça, rebarba, dimensão, tolerân..."
1,"Após ajuste no sensor de corrente, o sistema v...","[após, ajust, sensor, corr, sistem, volt, oper...","[após, ajustir, sensor, corrente, sistema, vol..."
2,A ANÁLISE DE CONSUMO PERMITIU REDUZIR DESPERDÍ...,"[anális, consum, permit, reduz, desperdíci, en...","[análise, consumo, permitir, reduzir, desperdí..."
3,O ALMOXARIFADO SEPAROU OS MATERIAIS NECESSÁRIO...,"[almoxarif, separ, mater, necess, ord, produç,...","[almoxarifado, separar, material, necessário, ..."
4,A otimização do setup reduziu o tempo de ciclo...,"[otimiz, setup, reduz, temp, cicl, aument, pro...","[otimização, setup, reduzir, tempo, ciclo, aum..."
5,A campanha de segurança reduziu ocorrências de...,"[campanh, seguranç, reduz, ocorr, quas, acid, ...","[campanha, segurança, reduzir, ocorrência, qua..."
6,Foram identificadas peças com rebarbas e dimen...,"[identific, peç, rebarb, dimens, toler, lot, l...","[identificar, peça, rebarba, dimensão, tolerân..."
7,"Após ajuste no sensor de corrente, o sistema v...","[após, ajust, sensor, corr, sistem, volt, oper...","[após, ajustir, sensor, corrente, sistema, vol..."
8,A análise de consumo permitiu reduzir desperdí...,"[anális, consum, permit, reduz, desperdíci, en...","[análise, consumo, permitir, reduzir, desperdí..."
9,"Após ajuste no sensor de corrente, o sistema v...","[após, ajust, sensor, corr, sistem, volt, oper...","[após, ajustir, sensor, corrente, sistema, vol..."


## 11. Transformando tokens em texto novamente

Junte os tokens processados em uma única string para facilitar usos futuros, como TF-IDF, embeddings ou classificação.

In [29]:
# TODO: transforme as listas de tokens em texto
df['texto_processado_stemming'] = df['pipeline_stemming'].apply(lambda tokens: ' '.join(tokens))
df['texto_processado_lematizacao'] = df['pipeline_lematizacao'].apply(lambda tokens: ' '.join(tokens))

df[['texto_original', 'texto_processado_stemming', 'texto_processado_lematizacao']].head(10)

,texto_original,texto_processado_stemming,texto_processado_lematizacao
0,Foram identificadas peças com rebarbas e dimen...,identific peç rebarb dimens toler lot l,identificar peça rebarba dimensão tolerância l...
1,"Após ajuste no sensor de corrente, o sistema v...",após ajust sensor corr sistem volt oper estabi...,após ajustir sensor corrente sistema voltar op...
2,A ANÁLISE DE CONSUMO PERMITIU REDUZIR DESPERDÍ...,anális consum permit reduz desperdíci energ pr...,análise consumo permitir reduzir desperdício e...
3,O ALMOXARIFADO SEPAROU OS MATERIAIS NECESSÁRIO...,almoxarif separ mater necess ord produç op sen...,almoxarifado separar material necessário ordem...
4,A otimização do setup reduziu o tempo de ciclo...,otimiz setup reduz temp cicl aument produt linh,otimização setup reduzir tempo ciclo aumentar ...
5,A campanha de segurança reduziu ocorrências de...,campanh seguranç reduz ocorr quas acid áre mont,campanha segurança reduzir ocorrência quase ac...
6,Foram identificadas peças com rebarbas e dimen...,identific peç rebarb dimens toler lot l urgent,identificar peça rebarba dimensão tolerância l...
7,"Após ajuste no sensor de corrente, o sistema v...",após ajust sensor corr sistem volt oper estabi...,após ajustir sensor corrente sistema voltar op...
8,A análise de consumo permitiu reduzir desperdí...,anális consum permit reduz desperdíci energ pr...,análise consumo permitir reduzir desperdício e...
9,"Após ajuste no sensor de corrente, o sistema v...",após ajust sensor corr sistem volt oper estabi...,após ajustir sensor corrente sistema voltar op...


## 12. Atividade guiada

### Situação-problema

A empresa Alpha-SP deseja criar futuramente um classificador automático de registros industriais.

Antes de treinar qualquer modelo, a equipe precisa entregar um dataset textual pré-processado, com duas versões:

1. texto processado com stemming;
2. texto processado com lematização.

### Tarefa

Gere um novo arquivo CSV com as colunas:

- `id`
- `texto_original`
- `categoria`
- `sentimento`
- `texto_processado_stemming`
- `texto_processado_lematizacao`

In [30]:
colunas_saida = [
    'id',
    'texto_original',
    'categoria',
    'sentimento',
    'texto_processado_stemming',
    'texto_processado_lematizacao'
]

# TODO: crie o dataframe de saída
df_saida = df[colunas_saida]

# TODO: visualize o resultado
df_saida.head(5)

,id,texto_original,categoria,sentimento,texto_processado_stemming,texto_processado_lematizacao
0,1,Foram identificadas peças com rebarbas e dimen...,qualidade,negativo,identific peç rebarb dimens toler lot l,identificar peça rebarba dimensão tolerância l...
1,2,"Após ajuste no sensor de corrente, o sistema v...",manutencao,positivo,após ajust sensor corr sistem volt oper estabi...,após ajustir sensor corrente sistema voltar op...
2,3,A ANÁLISE DE CONSUMO PERMITIU REDUZIR DESPERDÍ...,energia,positivo,anális consum permit reduz desperdíci energ pr...,análise consumo permitir reduzir desperdício e...
3,4,O ALMOXARIFADO SEPAROU OS MATERIAIS NECESSÁRIO...,logistica,neutro,almoxarif separ mater necess ord produç op sen...,almoxarifado separar material necessário ordem...
4,5,A otimização do setup reduziu o tempo de ciclo...,producao,positivo,otimiz setup reduz temp cicl aument produt linh,otimização setup reduzir tempo ciclo aumentar ...


In [31]:
# TODO: salve o arquivo CSV processado
df_saida.to_csv('aula12_dataset_textos_processados.csv', index=False, encoding='utf-8-sig')

print("Arquivo gerado com sucesso!")

Arquivo gerado com sucesso!


## 13. Questões para reflexão

Responda com suas palavras:

1. Qual é a diferença entre stemming e lematização?
2. Em qual situação a remoção de stopwords pode prejudicar uma análise textual?
3. Por que é importante limpar o texto antes da tokenização?
4. O que pode acontecer se removermos números em um problema industrial?
5. Para um chatbot com IA generativa, você usaria stemming, lematização ou nenhuma das duas técnicas? Justifique.

In [ ]:
# Respostas do aluno

resposta_1 = """


"""

resposta_2 = """


"""

resposta_3 = """


"""

resposta_4 = """


"""

resposta_5 = """


"""

print("Respostas registradas.")

## 14. Critérios de avaliação

| Critério | Atingiu | Não atingiu |
|---|---|---|
| Executa corretamente a limpeza textual | Realiza padronização, remoção de ruídos e tratamento básico do texto | Não realiza limpeza ou remove informações sem critério |
| Aplica tokenização corretamente | Divide o texto em tokens de forma adequada | Não consegue gerar tokens ou gera estrutura incoerente |
| Remove stopwords com coerência | Utiliza lista adequada e compreende limitações | Remove palavras sem critério técnico |
| Aplica stemming corretamente | Utiliza o RSLPStemmer e interpreta os radicais | Aplica técnica incorretamente ou não interpreta o resultado |
| Aplica lematização corretamente | Utiliza SpaCy e compara os lemas obtidos | Não executa ou não compreende a lematização |
| Organiza pipeline adequadamente | Cria sequência lógica de pré-processamento | Executa etapas soltas, sem organização |
| Demonstra coerência técnica | Justifica escolhas e reconhece limitações | Aplica técnicas sem análise crítica |